# 05 - Make the submission ZIP
Writes `README.md` and `requirements.txt`, then packs code, trained model, results and supporting files into `submission/DWC26_<team>.zip`. Run after notebooks 00-04 (save them first: File -> Save).

In [19]:
# SETUP - run this cell first, every time you open the notebook.
# It connects Google Drive, installs one extra package and loads our helper code.
import os, sys, subprocess, importlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/DWC26'          # the shared team folder
except ImportError:                                 # (only used when running outside Colab)
    BASE = os.environ.get('DWC_BASE', 'DWC26')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-calamine'], check=False, capture_output=True)
sys.path.insert(0, f'{BASE}/notebooks')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import dwc_utils as du; importlib.reload(du)
P = du.get_paths(BASE)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready. Project folder:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready. Project folder: /content/drive/MyDrive/DWC26


### Settings - fill in your team details

In [20]:
TEAM_NAME = '404 Team Not Found'
UNIVERSITY = 'UNIVERSITI KEBANGSAAN MALAYSIA'
MEMBERS = [                                   # (name, main contribution)
    ('Farid', 'Data preparation and exploration (notebooks 00-01)'),
    ('Alya', 'Proxy model (notebook 02)'),
    ('Aqil', 'History matching and forecast (notebooks 03-04)'),
    ('Iman', 'Power BI dashboard, documentation and video'),
]
OVERWRITE_README = True    # set to False after editing README.md by hand, so your edits are kept

### Step 0 - Final check of all outputs
Every line should start with OK. If any line says PROBLEM, fix that first (or send the output for help).

In [21]:
import glob
ok_all = True
def check(cond, msg):
    global ok_all
    print(('OK    ' if cond else 'PROBLEM ') + msg); ok_all &= bool(cond)

need = {'processed': ['params.parquet', 'sim_long.parquet', 'obs_long.parquet', 'grid_features.parquet',
                      'arrays.npz', 'proxy.joblib', 'best10_scaled.npy'],
        'outputs': ['DWC26_Deliverable.xlsx', 'best10_parameters.csv', 'proxy_metrics.csv',
                    'hindcast_results.csv', 'sensitivity.csv'],
        'powerbi': ['best10.csv', 'cases.csv', 'cashflow.csv', 'grid_case1.csv', 'hm_samples.csv',
                    'production_long.csv', 'proxy_metrics.csv']}
for folder, files in need.items():
    for f in files:
        check(os.path.exists(f"{BASE}/{folder}/{f}"), f'{folder}/{f} exists')
figs = sorted(os.path.basename(f) for f in glob.glob(f"{P['FIG']}/*.png"))
check(len(figs) == 10, f'outputs/figures has {len(figs)} of 10 charts {figs if len(figs) < 10 else ""}')
nbs = sorted(os.path.basename(f) for f in glob.glob(f"{BASE}/notebooks/0*.ipynb"))
check(len(nbs) >= 6 and os.path.exists(f"{BASE}/notebooks/dwc_utils.py"), f'notebooks: {nbs} + dwc_utils.py')

# parameters and production
params = pd.read_parquet(f"{P['PROC']}/params.parquet")
check(len(params) == 100 and params.case_id.tolist() == list(range(1, 101)), f'params: {len(params)} cases, numbered 1-100')
A = np.load(f"{P['PROC']}/arrays.npz")
check(A['oil_cum'].shape[0] == 100 and not np.isnan(A['oil_cum']).any(), f'arrays: oil_cum shape {A["oil_cum"].shape}, no gaps')
check(str(pd.to_datetime(A['grid'][-1]).date()) == '2008-01-01', f'history ends {pd.to_datetime(A["grid"][-1]).date()}')
obs_end = {v: A['obs_' + v][-1] for v in du.CUMS}
check(all(np.isfinite(list(obs_end.values()))), f'real-field 2008 totals: ' + ', '.join(f'{k} {v:,.0f}' for k, v in obs_end.items()))

# proxy
met = pd.read_csv(f"{P['OUT']}/proxy_metrics.csv")
import joblib; M = joblib.load(f"{P['PROC']}/proxy.joblib")
NAMES = {'gp': 'Gaussian Process', 'poly': 'Polynomial', 'forest': 'Random forest'}
t = met[(met.set == 'test') & (met.method == NAMES[M['method']])]
check(len(t) == 3 and (t['MAPE_end_%'] < 2).all(), f'proxy ({NAMES[M["method"]]}) test MAPE %: ' + ', '.join(f'{r.variable} {r["MAPE_end_%"]:.2f}' for _, r in t.iterrows()))

# best 10
b = pd.read_csv(f"{P['OUT']}/best10_parameters.csv")
inside = all(((b[p] >= params[p].min() * 0.999) & (b[p] <= params[p].max() * 1.001)).all() for p in du.PARAMS)
check(len(b) == 10 and b['rank'].tolist() == list(range(1, 11)) and b['misfit'].is_monotonic_increasing,
      'best10: 10 ranked rows, misfit increasing')
check(inside, 'best10: all parameters inside the ranges of the 100 cases')

# hindcast
hc = pd.read_csv(f"{P['OUT']}/hindcast_results.csv")
check(hc['hyperbolic_oil_err_%'].count() >= 50, f'hindcast: {hc["hyperbolic_oil_err_%"].count()} of 100 cases tested, median oil error {hc["hyperbolic_oil_err_%"].median():.2f}%')

# the submission file
import re
tpl_all = pd.read_excel(du.find_raw_files(P['RAW'])['09'], sheet_name=None, engine='calamine')
sheets = pd.read_excel(f"{P['OUT']}/DWC26_Deliverable.xlsx", sheet_name=None, engine='calamine')
case_tpl = sorted([n for n in tpl_all if re.findall(r'\d+', n)], key=lambda n: int(re.findall(r'\d+', n)[-1]))
if len(case_tpl) >= 10:                                   # template has one sheet per case
    expected = case_tpl[:10]; tpl_of = {n: tpl_all[n] for n in expected}
    check(list(sheets) == list(tpl_all), f'deliverable has the same sheets as the template: {list(sheets)}')
else:                                                     # single-sheet template
    expected = [f'Case_{k:02d}' for k in range(1, 11)]
    tpl_of = {n: list(tpl_all.values())[0] for n in expected}
    check(list(sheets) == expected, f'deliverable sheets: {list(sheets)}')
for name in expected:
    if name not in sheets:
        check(False, f'{name}: sheet missing'); continue
    s, t = sheets[name], tpl_of[name]
    vals = s.iloc[:, 1:4].astype(float)
    same_dates = len(s) == len(t) and s.iloc[:, 0].astype(str).tolist() == t.iloc[:, 0].astype(str).tolist()
    no_blank = not vals.isna().any().any()
    rising = bool((vals.diff().dropna() >= 0).all().all())
    first = dict(zip(['gas_cum', 'oil_cum', 'water_cum'], vals.iloc[0]))
    near = all(first[v] >= obs_end[v] and first[v] < obs_end[v] * 1.15 for v in du.CUMS)
    check(same_dates and no_blank and rising and near,
          f'{name}: {len(s)} rows (template {len(t)}), {s.iloc[0, 0]} to {s.iloc[-1, 0]} | same dates {same_dates}, '
          f'no blanks {no_blank}, increasing {rising}, starts just above 2008 totals {near}')
s1 = sheets[expected[0]]
print(f'\n{expected[0]} on {s1.iloc[-1, 0]}: gas {s1.iloc[-1, 1]:,.0f} MSCF, oil {s1.iloc[-1, 2]:,.0f} STB, water {s1.iloc[-1, 3]:,.0f} STB')
print('\nALL CHECKS PASSED' if ok_all else '\nSOME CHECKS FAILED - send this output to Claude')

OK    processed/params.parquet exists
OK    processed/sim_long.parquet exists
OK    processed/obs_long.parquet exists
OK    processed/grid_features.parquet exists
OK    processed/arrays.npz exists
OK    processed/proxy.joblib exists
OK    processed/best10_scaled.npy exists
OK    outputs/DWC26_Deliverable.xlsx exists
OK    outputs/best10_parameters.csv exists
OK    outputs/proxy_metrics.csv exists
OK    outputs/hindcast_results.csv exists
OK    outputs/sensitivity.csv exists
OK    powerbi/best10.csv exists
OK    powerbi/cases.csv exists
OK    powerbi/cashflow.csv exists
OK    powerbi/grid_case1.csv exists
OK    powerbi/hm_samples.csv exists
OK    powerbi/production_long.csv exists
OK    powerbi/proxy_metrics.csv exists
OK    outputs/figures has 10 of 10 charts 
OK    notebooks: ['00_load_data.ipynb', '01_explore.ipynb', '02_train_proxy.ipynb', '03_history_match.ipynb', '04_forecast_submit.ipynb', '05_make_submission (1).ipynb'] + dwc_utils.py
OK    params: 100 cases, numbered 1-100
OK  

### Step 1 - Collect the results

In [22]:
import joblib, shutil
M = joblib.load(f"{P['PROC']}/proxy.joblib")
NAMES = {'gp': 'Gaussian Process', 'poly': 'Polynomial', 'forest': 'Random forest'}
METHOD = NAMES[M['method']]
met = pd.read_csv(f"{P['OUT']}/proxy_metrics.csv")
test = met[(met['set'] == 'test') & (met['method'] == METHOD)].set_index('variable')
best = pd.read_csv(f"{P['OUT']}/best10_parameters.csv").iloc[0]
hc = pd.read_csv(f"{P['OUT']}/hindcast_results.csv")
score_h = hc['hyperbolic_oil_err_%'].median() + hc['hyperbolic_water_err_%'].median()
score_e = hc['exponential_oil_err_%'].median() + hc['exponential_water_err_%'].median()
HC_NAME = 'hyperbolic' if score_h <= score_e else 'exponential'
hc_oil, hc_wat = hc[f'{HC_NAME}_oil_err_%'].median(), hc[f'{HC_NAME}_water_err_%'].median()
prod = pd.read_csv(f"{P['PBI']}/production_long.csv", parse_dates=['date'])
fcst = prod[prod['source'] == 'forecast']
END_DATE = fcst['date'].max().date()                      # last forecast date (2028-01-01)
end = (fcst[fcst['date'] == fcst['date'].max()]
       .pivot(index='case', columns='variable', values='value')
       .rename(columns={'gas_cum': 'gas', 'oil_cum': 'oil', 'water_cum': 'water'}))
q = end.quantile([0.1, 0.5, 0.9])
npv_file = f"{P['PBI']}/best10.csv"
npv = pd.read_csv(npv_file)['NPV_musd'] if os.path.exists(npv_file) else None
fmt = lambda x: f'{x:,.0f}'
print('Method:', METHOD); display(test[['MAPE_end_%', 'R2']]); print(best); print(q)

Method: Gaussian Process


,MAPE_end_%,R2
variable,,
oil_cum,0.05417,0.9997
gas_cum,0.05383,0.9998
water_cum,0.1016,0.9998


rank                1
fault_trans      0.15
poro_mult       1.381
perm_mult       1.084
aquifer_pv        188
misfit        0.04653
Name: 0, dtype: float64
variable       gas       oil     water
0.1       2.08e+07 5.725e+07 4.467e+07
0.5      2.089e+07  5.75e+07 4.481e+07
0.9      2.095e+07 5.768e+07 4.491e+07


### Step 2 - Write README.md and requirements.txt

In [23]:
from importlib.metadata import version
pkgs = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'openpyxl', 'pyarrow', 'joblib', 'python-calamine']
req = [f'{p}=={version(p)}' for p in pkgs]
open(f"{BASE}/requirements.txt", 'w').write('\n'.join(req) + '\n')

npv_line = (f"| NPV 2008-2028, P10 / P50 / P90 (assumed prices, notebook 04) | "
            f"{np.percentile(npv, 10):,.0f} / {np.percentile(npv, 50):,.0f} / {np.percentile(npv, 90):,.0f} million USD |"
            if npv is not None else '')
members = '\n'.join(f'| {n} | {c} |' for n, c in MEMBERS)
readme = f"""# DataWorks Challenge 2026 - ML-Assisted History Matching and 20-Year Production Forecast

**Team:** {TEAM_NAME}  |  **University:** {UNIVERSITY}

## 1. Problem
An oil field has 10 years of production history (1998-2008). Four subsurface properties are uncertain:
fault transmissibility, porosity multiplier, permeability multiplier and aquifer pore volume.
100 reservoir-simulator runs with different values are provided. We (1) build a fast ML proxy of the simulator,
(2) find the parameter values that reproduce the observed history, (3) forecast oil, gas and water production
to 2028 for the 10 best cases, and (4) estimate whether the field makes money.

## 2. Approach
| Step | Notebook | Method |
| --- | --- | --- |
| Data preparation | `00_load_data` | Reads all Excel files; fixes US dates, duplicate report dates, blanks and -999.25 nulls; summarises each grid |
| Exploration | `01_explore` | Common quarterly time grid; sensitivity analysis; data checks |
| Proxy model | `02_train_proxy` | PCA + regression (Gaussian Process vs polynomial vs random forest); chosen on validation cases 71-85, scored on test cases 86-100, retrained on all 100 |
| History matching | `03_history_match` | Normalised least-squares misfit; Sobol scan (16,384 sets) + differential evolution; 10 diverse best sets via K-means |
| Forecast | `04_forecast_submit` | Arps decline of liquid rate x logistic water cut vs cumulative oil; validated by a 2005-2008 hindcast; anchored to observed 2008 totals; gas = oil x GOR |
| Economics | `04_forecast_submit` | Yearly cash flow and NPV at 10% discount (assumed prices) |
| Packaging | `05_make_submission` | Writes this README and the ZIP |

Shared functions are in `notebooks/dwc_utils.py`.

## 3. Results
| Item | Value |
| --- | --- |
| Proxy method | {METHOD} |
| Proxy test error at 2008 (MAPE) | oil {test.loc['oil_cum', 'MAPE_end_%']:.2f}%, gas {test.loc['gas_cum', 'MAPE_end_%']:.2f}%, water {test.loc['water_cum', 'MAPE_end_%']:.2f}% |
| Proxy test R2 | oil {test.loc['oil_cum', 'R2']:.4f}, gas {test.loc['gas_cum', 'R2']:.4f}, water {test.loc['water_cum', 'R2']:.4f} |
| Best-match parameters | fault transmissibility {best['fault_trans']:.4f}, porosity multiplier {best['poro_mult']:.4f}, permeability multiplier {best['perm_mult']:.4f}, aquifer pore volume {best['aquifer_pv']:.2f} |
| Forecast method (hindcast median error 2005-2008) | {HC_NAME} decline: oil {hc_oil:.2f}%, water {hc_wat:.2f}% |
| Cumulative oil by {END_DATE}, P10 / P50 / P90 | {fmt(q.loc[0.1, 'oil'])} / {fmt(q.loc[0.5, 'oil'])} / {fmt(q.loc[0.9, 'oil'])} STB |
| Cumulative gas by {END_DATE}, P10 / P50 / P90 | {fmt(q.loc[0.1, 'gas'])} / {fmt(q.loc[0.5, 'gas'])} / {fmt(q.loc[0.9, 'gas'])} MSCF |
| Cumulative water by {END_DATE}, P10 / P50 / P90 | {fmt(q.loc[0.1, 'water'])} / {fmt(q.loc[0.5, 'water'])} / {fmt(q.loc[0.9, 'water'])} STB |
{npv_line}

**Submission file:** `outputs/DWC26_Deliverable.xlsx` (one sheet per case in the template's order; the first case sheet = best match).
**Trained model:** `processed/proxy.joblib`.

## 4. Folder structure
```
DWC26/
  README.md                 this file
  requirements.txt          Python packages and versions used
  notebooks/                source code: 00-05 notebooks (with outputs) + dwc_utils.py
  processed/                trained model (proxy.joblib) and intermediate data
  outputs/                  DWC26_Deliverable.xlsx, result tables (CSV), figures/
  raw/                      template file; put the competition data files here to re-run notebook 00
```

## 5. How to run
**Option A - view only:** open the notebooks; all outputs (tables, charts) are saved inside them.

**Option B - re-run notebooks 01-04 without the raw data:** the intermediate files in `processed/` are included.
1. Upload the `DWC26` folder to Google Drive so that it is at `My Drive/DWC26`.
2. Open a notebook in Google Colab and choose Runtime -> Run all. Run 01 -> 02 -> 03 -> 04 in order.

**Option C - full run from the raw data:** put the 9 competition Excel files in `raw/` and run 00 -> 04.

**Running locally instead of Colab:** Python 3.10+, `pip install -r requirements.txt`, set the environment
variable `DWC_BASE` to the full path of the `DWC26` folder, then start Jupyter and run the notebooks in order.

## 6. Key settings and assumptions
- Train / validation / test split: cases 1-70 / 71-85 / 86-100.
- Misfit: oil, gas and water cumulative, allowed error 3% of each series' maximum.
- Economics (assumptions, not provided by the organisers): oil USD 70/bbl, gas USD 3/MSCF,
  water handling USD 2/bbl, fixed operating cost USD 60 million/year, discount rate 10%.
- Random seeds are fixed, so results are reproducible.

## 7. Team
| Member | Contribution |
| --- | --- |
{members}
"""
if OVERWRITE_README or not os.path.exists(f"{BASE}/README.md"):
    open(f"{BASE}/README.md", 'w').write(readme)
    print('README.md written to', BASE)
else:
    print('README.md kept as you edited it')
print('requirements.txt:', req)

README.md written to /content/drive/MyDrive/DWC26
requirements.txt: ['numpy==2.1.3', 'pandas==2.2.3', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'openpyxl==3.1.5', 'pyarrow==23.0.1', 'joblib==1.6.0', 'python-calamine==0.8.3']


### Step 3 - Build the ZIP

In [24]:
ROOT = f"{BASE}/submission"
SUB = f"{ROOT}/DWC26"
if os.path.exists(SUB): shutil.rmtree(SUB)                          # start clean
os.makedirs(f"{SUB}/notebooks"); os.makedirs(f"{SUB}/processed"); os.makedirs(f"{SUB}/raw")
for f in sorted(os.listdir(f"{BASE}/notebooks")):                     # source code
    if f.endswith('.ipynb') or f == 'dwc_utils.py':
        shutil.copy(f"{BASE}/notebooks/{f}", f"{SUB}/notebooks/{f}")
for f in ['proxy.joblib', 'best10_scaled.npy', 'arrays.npz', 'params.parquet',
          'sim_long.parquet', 'obs_long.parquet', 'grid_features.parquet']:   # model + intermediate data
    if os.path.exists(f"{P['PROC']}/{f}"):
        shutil.copy(f"{P['PROC']}/{f}", f"{SUB}/processed/{f}")
shutil.copytree(P['OUT'], f"{SUB}/outputs")                          # results and figures
FILES = du.find_raw_files(P['RAW'])
if '09' in FILES: shutil.copy(FILES['09'], f"{SUB}/raw/")             # template (small), needed by notebook 04
open(f"{SUB}/raw/PUT_COMPETITION_DATA_HERE.txt", 'w').write(
    'Place the 9 competition Excel files (01-09) here to re-run notebook 00.\n')
for f in ['README.md', 'requirements.txt']:
    shutil.copy(f"{BASE}/{f}", f"{SUB}/{f}")
safe = ''.join(ch if ch.isalnum() else '_' for ch in TEAM_NAME).strip('_')
zip_path = shutil.make_archive(f"{ROOT}/DWC26_{safe}", 'zip', root_dir=ROOT, base_dir='DWC26')
for folder, _, files in sorted(os.walk(SUB)):                         # show what is inside
    depth = folder[len(SUB):].count('/')
    print('  ' * depth + os.path.basename(folder) + '/')
    for f in sorted(files):
        print('  ' * (depth + 1) + f)
print(f'\nZIP ready: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')

DWC26/
  README.md
  requirements.txt
  notebooks/
    00_load_data.ipynb
    01_explore.ipynb
    02_train_proxy.ipynb
    03_history_match.ipynb
    04_forecast_submit.ipynb
    05_make_submission (1).ipynb
    dwc_utils.py
  outputs/
    DWC26_Dashboard.pbix
    DWC26_Deliverable.xlsx
    best10_parameters.csv
    hindcast_results.csv
    proxy_metrics.csv
    sensitivity.csv
    figures/
      01_fan_cumulative.png
      02_fan_rates.png
      03_parameter_coverage.png
      04_sensitivity.png
      05_grid_map.png
      06_proxy_test_cases.png
      07_closest_real_cases.png
      08_history_match.png
      09_parameter_uncertainty.png
      10_forecast.png
  processed/
    arrays.npz
    best10_scaled.npy
    grid_features.parquet
    obs_long.parquet
    params.parquet
    proxy.joblib
    sim_long.parquet
  raw/
    09 Template Deliverable.xlsx
    PUT_COMPETITION_DATA_HERE.txt

ZIP ready: /content/drive/MyDrive/DWC26/submission/DWC26_404_Team_Not_Found.zip  (6.4 MB)
